# Exercise 1.3 — Predict handwritten digits (MNIST, dense NN)

Build a classifier that recognises handwritten digits — 28×28-pixel grayscale images,
10 classes. The compute needed to *run* this model is well within reach of the
microcontroller in your kit.

**MNIST facts:** 70,000 images (60k train / 10k test) of digits written by many different
people; the "Hello World" benchmark of image classification. The model here is a
two-layer dense network with ~16k parameters.

Contrast this with Exercise 1.2: there, humans engineered the features (rooms, income, …).
Here the input is **raw pixels** — all the reasoning about content is left to the network.
No human intuition used. Remember this trade-off; it returns on Day 2 when we choose
between hand-crafted features and raw-signal models on the fan.

Fill in the `TODO`s. If you would rather follow a ready-made walkthrough, the original
version of this exercise lives here:
[TF_MNIST_Classification_v2.ipynb (UNIFEI-IESTI01)](https://colab.research.google.com/github/Mjrovai/UNIFEI-IESTI01-TinyML-2022.1/blob/main/00_Curse_Folder/1_Fundamentals/Class_09/TF_MNIST_Classification_v2.ipynb)

In [ ]:
!pip install tensorflow matplotlib numpy scikit-learn

## Step 1 — Load and inspect the data

MNIST ships with Keras: `keras.datasets.mnist.load_data()` returns
`(x_train, y_train), (x_test, y_test)`.

**Questions:**

- What are the shapes of `x_train` and `y_train`?
- What is the dtype and the value range of the pixels?
- How many images per class — is the dataset balanced?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

# TODO: load the MNIST dataset into (x_train, y_train), (x_test, y_test)

# TODO: print the shapes, dtype and min/max of the training images

# TODO: print how many samples there are per class (hint: np.bincount)

## Step 2 — Look at the data before you model it

Plot the first 25 training images in a 5×5 grid with their labels as titles.

Never train on data you have not looked at. Half the "the model doesn't learn" problems
in this course are visible with `imshow` and thirty seconds of attention.

In [ ]:
# TODO: plot a 5x5 grid of training images, each titled with its label
# hint: plt.subplots(5, 5, figsize=(8, 8)), ax.imshow(img, cmap="gray"), ax.axis("off")

## Step 3 — Normalise

The pixels are `uint8` in `[0, 255]`. Neural networks train far better on small,
centred inputs — scale both the training and the test images to floats in `[0, 1]`.

**Question:** why does dividing by 255 help, when the network could in principle learn
the same scaling in its first layer's weights?

In [ ]:
# TODO: scale x_train and x_test to float32 in [0, 1]

## Step 4 — Build and train the model

The architecture:

| Layer | Why |
|---|---|
| `Flatten(input_shape=(28, 28))` | 28×28 image → a 784-long vector; dense layers have no notion of 2-D |
| `Dense(20, activation="relu")` | the hidden layer that does the work |
| `Dense(10, activation="softmax")` | one output per digit, turned into probabilities |

Compile with `optimizer="adam"`, `loss="sparse_categorical_crossentropy"`
(*sparse* because the labels are integers `0-9`, not one-hot vectors) and
`metrics=["accuracy"]`. Train for ~10 epochs with `validation_split=0.1`.

Run `model.summary()` and check the parameter count against the formula
`inputs × outputs + outputs` for each dense layer.

**Expected output:** test accuracy ≈ **97–98 %** after a few epochs.

In [ ]:
# TODO: build the Sequential model

# TODO: compile it

# TODO: print model.summary() and verify the ~16k parameters by hand

# TODO: fit it for ~10 epochs with validation_split=0.1, keeping the History object
# history = model.fit(...)

## Step 5 — Evaluate and plot the learning curves

Evaluate on the test set, then plot training vs. validation accuracy (and loss) per epoch.

**Questions:**

- Is the test accuracy close to the validation accuracy? Should it be?
- Where do the two curves start to diverge, and what is that called?

In [ ]:
# TODO: evaluate on the test set and print loss + accuracy

# TODO: plot history.history["accuracy"] vs ["val_accuracy"] (and the losses)

## Step 6 — Break it on purpose

The learning rate is the first knob to touch when training misbehaves. Rebuild and
retrain the *same* model twice, with an explicit optimizer:

```python
keras.optimizers.Adam(learning_rate=...)
```

1. **×10** the default (`0.001` → `0.01`)
2. **÷100** the default (`0.001` → `0.00001`)

Plot all three loss curves on one figure.

**Expected output:** LR ×10 — the loss jumps around or explodes. LR ÷100 — the loss
decreases painfully slowly. Sanctioned vandalism: this is the fastest way to learn what
a loss curve is telling you.

In [ ]:
# TODO: write a small helper that builds + compiles the model for a given learning rate,
#       so you are not copy-pasting the architecture three times
# def build_and_train(lr, epochs=10): ...

# TODO: train with lr = 0.001 (default), 0.01 and 0.00001

# TODO: plot the three loss curves on one figure, with a legend

## Step 7 — Confusion matrix

Accuracy is one number and it hides *which* mistakes the model makes. Predict the test
set, take the `argmax` of each prediction, and build the confusion matrix
(`sklearn.metrics.confusion_matrix`, then `ConfusionMatrixDisplay` or `plt.imshow`).

**Questions:**

- Which digit pairs get confused? Do the mistakes look human?
- Plot a handful of misclassified images — would *you* have got them right?
- What does the matrix show that accuracy hides? (Keep this in mind for Day 3: on the
  fan, a class that never appears in training is *always* misclassified, and accuracy
  will not tell you.)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TODO: predict the test set and convert the probabilities to predicted class indices

# TODO: build and plot the confusion matrix

# TODO: plot a few misclassified images with "true vs predicted" as the title

## Step 8 — Save the model and open it in Netron

Save the trained model and inspect it in [Netron](https://netron.app/) (drag the file
into the browser — nothing is uploaded).

**Questions:**

- Where do the ~16k parameters come from? Verify layer by layer with
  `inputs × outputs + outputs`.
- What are the tensor shapes between layers?
- At 4 bytes per float32 parameter, how much flash would the weights alone need? Compare
  with the 1 MB flash / 256 KB RAM of the RAK4631 in your kit. This is the arithmetic
  the whole course is about.

In [ ]:
# TODO: save the model (model.save("mnist_dense.keras"), or .h5 if Netron prefers it)
# then download it and open it at https://netron.app/

## Observe & discuss

- What happens when you adjust the learning rate up or down before training?
- What is the purpose of a confusion matrix — what does it show that accuracy hides?
- The network was given raw pixels and no hints. What did it have to learn that a
  feature-engineering approach would have been *told*?
- Would this model fit on your RAK4631? What would you have to change to be sure?